In [3]:
import numpy as np
import pandas as pd
import xgboost as xgb
from sklearn.datasets import make_classification
from sklearn.feature_selection import VarianceThreshold, mutual_info_classif, f_classif
from sklearn.preprocessing import MinMaxScaler

from utils.data_processing import load_data, standarize, variance_threshold, correlation_filter, select_from_model, select_k_best

# 1. Generate a mock dataset: 5000 observations, 500 features
# 20 informative features, some redundant (collinear) features, and many useless ones.
X, y, XfinalTest = load_data()

print(f"Original dataset shape: {X.shape}")

Original dataset shape: (5000, 500)


In [18]:
y["y"]

0       0
1       1
2       0
3       1
4       0
       ..
4995    1
4996    1
4997    1
4998    1
4999    0
Name: y, Length: 5000, dtype: int64

In [20]:
# --- 1. Variance Threshold ---
# Drop features where 99% of the values are the same
var_thresholder = VarianceThreshold(threshold=(.99 * (1 - .99)))
var_thresholder.fit(X)
X_var_filtered = X.loc[:, var_thresholder.get_support()]
print(f"Features after dropping low variance: {X_var_filtered.shape[1]}")

# --- 2. Collinearity (Correlation) Filter ---
def drop_collinear_features(df, target, threshold=0.85):
    """
    Finds highly correlated pairs and drops the one with lower 
    absolute correlation to the target.
    """
    # Calculate absolute correlation matrix of features
    corr_matrix = df.corr().abs()
    
    # Upper triangle of correlation matrix
    upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
    
    # Find features with correlation greater than threshold
    to_drop = set()
    for column in upper.columns:
        highly_correlated_with = upper.index[upper[column] > threshold].tolist()
        for correlated_col in highly_correlated_with:
            # Check which one has lower correlation with the target
            corr_col1 = abs(df[column].corr(target["y"]))
            corr_col2 = abs(df[correlated_col].corr(target["y"]))
            
            if corr_col1 > corr_col2:
                to_drop.add(correlated_col)
            else:
                to_drop.add(column)
                
    return df.drop(columns=list(to_drop))

X_filtered = drop_collinear_features(X_var_filtered, y, threshold=0.85)
print(f"Features after dropping highly collinear ones: {X_filtered.shape[1]}")

Features after dropping low variance: 500
Features after dropping highly collinear ones: 489


In [23]:
y = y.to_numpy().ravel()

In [24]:
# Initialize a dataframe to hold our scores
scores_df = pd.DataFrame(index=X_filtered.columns)

# --- Metric 1: ANOVA F-statistic (Linear) ---
# f_classif returns F-values and p-values; we want the F-values
f_values, _ = f_classif(X_filtered, y)
scores_df['ANOVA'] = f_values

# --- Metric 2: Mutual Information (Non-linear) ---
# random_state ensures reproducibility
mi_values = mutual_info_classif(X_filtered, y, random_state=42)
scores_df['Mutual_Info'] = mi_values

# --- Metric 3: XGBoost Feature Importance (Tree-based) ---
# Train a quick, shallow XGBoost model to get feature importances
xgb_model = xgb.XGBClassifier(
    n_estimators=100, 
    max_depth=4, 
    eval_metric='logloss',
    random_state=42,
    n_jobs=-1
)
xgb_model.fit(X_filtered, y)

# We use 'gain' as it is the most robust measure of importance for tree splits
importance_dict = xgb_model.get_booster().get_score(importance_type='gain')
# Map scores back, filling 0 for features that were never used in a split
scores_df['XGB_Gain'] = scores_df.index.map(lambda x: importance_dict.get(x, 0.0))

scores_df.head()

,ANOVA,Mutual_Info,XGB_Gain
V1,0.874735,0.000000,4.802160
V2,2.601593,0.002295,7.628243
V3,4.811795,0.003667,5.664058
V4,0.607565,0.000211,0.000000
V5,11.360246,0.017163,11.352443


In [25]:
# Normalize the scores to a 0-1 scale so they carry equal weight
scaler = MinMaxScaler()
normalized_scores = scaler.fit_transform(scores_df)
normalized_scores_df = pd.DataFrame(
    normalized_scores, 
    columns=scores_df.columns, 
    index=scores_df.index
)

# Calculate the average score across all three metrics
normalized_scores_df['Average_Score'] = normalized_scores_df.mean(axis=1)

# Sort by the average score in descending order
final_ranking = normalized_scores_df.sort_values(by='Average_Score', ascending=False)

# Select the top 20 features
N_FEATURES_TO_SELECT = 20
top_20_features = final_ranking.head(N_FEATURES_TO_SELECT).index.tolist()

print(f"Top {N_FEATURES_TO_SELECT} Selected Features:")
display(final_ranking.head(N_FEATURES_TO_SELECT))

# Create your final, reduced dataset
X_final = X_filtered[top_20_features]
print(f"\nFinal dataset shape ready for XGBoost modeling: {X_final.shape}")

Top 20 Selected Features:


,ANOVA,Mutual_Info,XGB_Gain,Average_Score
V224,0.915863,0.548614,0.941731,0.802069
V390,1.000000,0.398320,0.397023,0.598448
V199,0.763122,0.383537,0.474093,0.540251
V32,0.820075,0.301604,0.495221,0.538967
V160,0.018192,0.850053,0.728867,0.532371
V345,0.682781,0.433490,0.403458,0.506576
V46,0.176901,0.316521,1.000000,0.497807
V5,0.172722,0.803252,0.446207,0.474060
V60,0.509820,0.535504,0.328093,0.457806
V380,0.007551,0.798475,0.508954,0.438327



Final dataset shape ready for XGBoost modeling: (5000, 20)
